# MNI MedSAM3: two-subject pilot
Updated in the same notebook. Sign in as **filippofocaccia2@gmail.com**, select **A100** (80GB preferred), and run all cells.

**Exactly one experiment:** subjects s23+s03, four L/R crops; fold0, seed0; same ten validation crops,
native MedSAM3 loss, medical LoRA initialization, 200 updates, accumulation2, warmup20, LR5e-5→5e-6, no augmentation.
The completed one-subject results remain available for the comparison at the end.

Reuses the MSD/ADNI framework: deterministic CPU transform caching, immutable background recovery uploads,
memory admission checks, live GPU telemetry, and checkpoint-specific evaluation feature reuse.
Fast evaluation must match the original full-volume predictions and log-probabilities and show a measured
speed gain; each subsequent volume has a checked slice. Failure automatically uses the original evaluator.
Three prompts share one image encoding per slice during audited inference. Training's batch size and GPU graph are unchanged.

Only this n2 run is scheduled, so the resource-aware worker limit is one. No extra subjects, seeds or folds are launched.
Run only one VM for this study. Interrupt the Run cell once and wait for all recovery uploads before disconnecting.
Background recovery snapshots are attempted every180seconds; abrupt VM loss can replay work since the latest verified snapshot.
New MNI acceleration has CPU tests; its A100 equivalence and speed are checked when you run it.


In [ ]:
import subprocess
subprocess.run(['nvidia-smi'],check=True)


In [ ]:
from google.colab import drive, auth
drive.mount('/content/drive')
auth.authenticate_user()
import google.auth
from googleapiclient.discovery import build
credentials, _ = google.auth.default()
service = build('drive', 'v3', credentials=credentials, cache_discovery=False)
email = service.about().get(fields='user(emailAddress)').execute()['user']['emailAddress']
if email.lower() != 'filippofocaccia2@gmail.com':
    raise RuntimeError('Authenticate with filippofocaccia2@gmail.com; got '+email)
from pathlib import Path
import hashlib, json, os, shutil, signal, subprocess, sys, zipfile
ROOT = Path('/content/hippo_mni_medsam3_n2_optimized')
BUNDLE = ROOT/'bundle'
SHARED = ROOT/'shared'
HOME = Path('/content/drive/MyDrive/hippo_medsam3_mni_pilot')
OLD = Path('/content/drive/MyDrive/hippo_medsam3_msd_extension/input')
RESULTS = HOME/'results_n2_f0s0_native200_optimized_v1'
OUTPUT = ROOT/'work'
RESULTS.mkdir(parents=True, exist_ok=True)
def sha(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(8*1024**2), b''): h.update(block)
    return h.hexdigest()
def copy_verified(source, target):
    target.parent.mkdir(parents=True, exist_ok=True)
    temp = target.with_name(target.name+'.uploading')
    shutil.copyfile(source, temp)
    if sha(source) != sha(temp): raise IOError('Copy mismatch: '+str(source))
    temp.replace(target)
print('Account:', email, '\nResults:', RESULTS)


In [ ]:
PACKAGE_SHA = '6fd2280e84a9fb5d0fefa88298428b1b1624af9a871a61c4619c0a579a74118a'
MANIFEST_SHA = 'cd48e0b2aa63f7128a030a57baabdbf17f9eca4f4fdbb216ca2c6bef676882c4'
OLD_PACKAGE_SHA = '4fef5c8191d394594f33c701bbf5e4c733a20cbfd5e8e2f2162a4efa42b3948b'
package = HOME/'medsam3_mni_n2_optimized.zip'
old_package = OLD/'medsam3_msd_108_fast_eval_code.zip'
for file, expected in [(package, PACKAGE_SHA), (old_package, OLD_PACKAGE_SHA)]:
    if sha(file) != expected: raise ValueError('Input ZIP differs: '+str(file))
def extract(archive_path, select, target):
    with zipfile.ZipFile(archive_path) as archive:
        for info in archive.infolist():
            name = Path(info.filename)
            if name.is_absolute() or '..' in name.parts or (info.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError('Unsafe ZIP member')
            relative = select(name)
            if relative is None or info.is_dir(): continue
            destination = target/relative
            value = archive.read(info)
            if destination.exists():
                if sha(destination) != hashlib.sha256(value).hexdigest():
                    raise ValueError('Existing input differs; retained: '+str(destination))
                continue
            destination.parent.mkdir(parents=True, exist_ok=True)
            destination.write_bytes(value)
extract(package, lambda p: p if p.parts[0]=='bundle' else None, ROOT)
extract(old_package, lambda p: Path(*p.parts[1:]) if len(p.parts)>2 and p.parts[0]=='bundle'
        and p.parts[1] in ('upstream','compatibility','weights') and p.name!='sam3.pt' else None, SHARED)
if sha(BUNDLE/'manifest.json') != MANIFEST_SHA: raise ValueError('Manifest mismatch')
for name, expected in json.loads((BUNDLE/'manifest.json').read_text()).items():
    if sha(BUNDLE/name) != expected: raise ValueError('Bundle changed: '+name)
protocol = json.loads((BUNDLE/'protocol.json').read_text())
BASE = ROOT/'sam3.pt'
previous_base = Path('/content/hippo_mni_medsam3/sam3.pt')
if previous_base.exists() and sha(previous_base)==protocol['base_sha256']: BASE=previous_base
if not BASE.exists(): copy_verified(OLD/'sam3.pt', BASE)
if sha(BASE) != protocol['base_sha256']: raise ValueError('Base weights differ')
if sha(SHARED/'weights/best_lora_weights.pt') != protocol['lora_sha256']: raise ValueError('Medical adapter differs')
print('Verified source, data and weights. Training:', protocol['train_cases'])


In [ ]:
VENV = ROOT/'venv'
previous_venv=Path('/content/hippo_mni_medsam3/venv')
if (previous_venv/'bin/python').exists(): VENV=previous_venv
PYTHON = VENV/'bin/python'
def setup(args):
    result = subprocess.run(list(map(str,args)), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    with (RESULTS/'setup.log').open('a') as log: log.write(result.stdout)
    print('\n'.join(result.stdout.splitlines()[-25:]))
    if result.returncode: raise RuntimeError('Setup failed; see setup.log')
if not PYTHON.exists(): setup([sys.executable,'-m','venv','--without-pip','--system-site-packages',VENV])
if subprocess.run([str(PYTHON),'-m','pip','--version'],capture_output=True).returncode:
    import urllib.request
    bootstrap = ROOT/'get-pip.py'
    urllib.request.urlretrieve('https://bootstrap.pypa.io/get-pip.py',bootstrap)
    setup([PYTHON,bootstrap])
setup([PYTHON,'-m','pip','install','--disable-pip-version-check','-r',BUNDLE/'requirements-colab.txt'])
env = dict(os.environ, PYTHONPATH=str(SHARED/'compatibility')+':'+str(BUNDLE/'source')+':'+str(BUNDLE/'source/scripts'),
    MNI_DURABLE=str(RESULTS), MNI_MANIFEST_SHA=MANIFEST_SHA, PYTHONUNBUFFERED='1',
    OMP_NUM_THREADS='4', MKL_NUM_THREADS='4', WANDB_MODE='disabled')
probe = subprocess.run([str(PYTHON),'-c',
    "import torch,json; assert torch.cuda.is_available() and torch.cuda.is_bf16_supported(); p=torch.cuda.get_device_properties(0); assert p.total_memory>=38*1024**3; print(json.dumps(dict(gpu=p.name,torch=torch.__version__,cuda=torch.version.cuda)))"],
    env=env, capture_output=True,text=True,check=True)
environment = json.loads(probe.stdout.strip().splitlines()[-1])
binding = dict(manifest=MANIFEST_SHA, **environment)
binding_path = RESULTS/'binding.json'
if binding_path.exists() and json.loads(binding_path.read_text()) != binding:
    raise RuntimeError('Resume environment changed; use the same GPU/software for this pilot')
binding_path.write_text(json.dumps(binding,indent=2))
# Archive the complete source, protocol, folds and data hashes before training.
for name in ['protocol.json','plan.json','saved_folds.json','data_audit.json','source_provenance.json','manifest.json']:
    copy_verified(BUNDLE/name, RESULTS/'provenance'/name)
copy_verified(HOME/'medsam3_mni_n2_optimized.zip', RESULTS/'provenance/medsam3_mni_n2_optimized.zip')
print(environment)


## Run the two-subject pilot

The existing MSD GPU controller prints usage and progress every30seconds. Run the results cell afterward to compare against n1 and complete verified cleanup.


In [ ]:
command=[str(PYTHON),'-m','scripts.colab_mni_medsam3.supervisor',
    '--root',str(ROOT),'--durable',str(RESULTS),'--base',str(BASE),'--max-workers','0','--hours','12']
process=subprocess.Popen(command,cwd=BUNDLE/'source',env=env,stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,text=True,bufsize=1)
with (RESULTS/'supervisor.log').open('a') as log:
    while True:
        try:
            for line in process.stdout:
                print(line,end='',flush=True);log.write(line);log.flush()
            code=process.wait();break
        except KeyboardInterrupt:
            print('Clean stop requested; wait for verified Drive recovery.')
            if process.poll() is None:process.send_signal(signal.SIGINT)
if code:raise RuntimeError('Run stopped or failed. Inspect the logs; preserve recovery. Rerun to resume after resolving the cause.')
done=json.loads((RESULTS/'completion.json').read_text())
print('Completed two-subject pilot:',done)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
evaluation = json.loads((RESULTS/'evaluation.json').read_text())
history = json.loads((RESULTS/'training_history.json').read_text())
expected = {(s,c) for s,cases in [('training_support',protocol['train_cases']),('validation',protocol['evaluation_cases'])] for c in cases}
if {(r['split'],r['case_name']) for r in evaluation['records']} != expected or len(history)!=200:
    raise ValueError('Incomplete evaluation/history; retain checkpoints')
if sha(RESULTS/'evaluation.json') != done['evaluation_sha256']:
    raise ValueError('Evaluation checksum mismatch')
rows=[]
for record in evaluation['records']:
    matrix=np.asarray(record['confusion'])
    tp=matrix[1:,1:].sum(); fp=matrix[0,1:].sum(); fn=matrix[1:,0].sum()
    row=dict(split=record['split'],case=record['case_name'],
             mean_subfield_dice=record['macro_hard_dice'],whole_hippocampus_dice=2*tp/(2*tp+fp+fn),
             foreground_fp=int(fp),foreground_fn=int(fn),
             subfield_confusions=int(matrix[1:,1:].sum()-np.trace(matrix[1:,1:])))
    for name, metrics in record['per_class'].items():
        if name=='background': continue
        for metric in ('dice','fp','fn'): row[name+'_'+metric]=metrics[metric]
    rows.append(row)
table=pd.DataFrame(rows); table.to_csv(RESULTS/'per_case_metrics.csv',index=False)
summary=table.groupby('split').mean(numeric_only=True)
summary.to_csv(RESULTS/'summary.csv')
display(summary[[c for c in summary if 'dice' in c]]*100)
fig,axes=plt.subplots(1,3,figsize=(15,4))
axes[0].plot([h['update'] for h in history],[h['native'] for h in history]); axes[0].set_title('Native training loss')
axes[1].plot([h['update'] for h in history],[h['gradient_norm'] for h in history]); axes[1].set_title('Gradient norm')
summary[[c+'_dice' for c in protocol['classes'][1:]]].T.plot.bar(ax=axes[2],ylim=(0,1),rot=25)
axes[2].set_title('Final subfield Dice');fig.tight_layout();fig.savefig(RESULTS/'pilot_results.png',dpi=160);plt.show()
# Check readable durable metadata before removing only this completed pilot's checkpoints.
metadata={p.name:sha(p) for p in RESULTS.iterdir() if p.is_file() and p.suffix in ('.json','.csv','.png','.log','.jsonl') and p.name not in ('archive_manifest.json','cleanup.json')}
manifest_path=RESULTS/'archive_manifest.json'
manifest_path.write_text(json.dumps(metadata,indent=2))
for name,expected_sha in json.loads(manifest_path.read_text()).items():
    if sha(RESULTS/name)!=expected_sha: raise ValueError('Archive verification failed')
    if name.endswith('.json'): json.loads((RESULTS/name).read_text())
deleted=[]
for checkpoint in [OUTPUT/'checkpoint_latest.pt',RESULTS/'checkpoint_latest.pt']:
    if checkpoint.exists():
        if sha(checkpoint)!=done['checkpoint_sha256']: raise ValueError('Checkpoint changed; retained')
        deleted.append(dict(path=str(checkpoint),bytes=checkpoint.stat().st_size,sha256=sha(checkpoint)))
        checkpoint.unlink()
cleanup_path=RESULTS/'cleanup.json'
if not cleanup_path.exists(): cleanup_path.write_text(json.dumps(dict(status='complete',deleted=deleted),indent=2))
print('Results archived and verified:', RESULTS)
# Retire only the verified final snapshot for this completed experiment.
import tarfile
recovery=RESULTS/'recovery'
marker=recovery/'latest.json'
if marker.exists():
    record=json.loads(marker.read_text());name=record['file']
    if Path(name).name!=name or not name.startswith('recovery-'):raise ValueError('Unsafe recovery path')
    archive=recovery/name
    if sha(archive)!=record['sha256']:raise ValueError('Recovery checksum mismatch; retained')
    with tarfile.open(archive) as tar:
        snapshot=json.load(tar.extractfile('snapshot_manifest.json'))
    if snapshot['checkpoint_latest.pt']!=done['checkpoint_sha256']:raise ValueError('Recovery is from another checkpoint; retained')
    cleanup=json.loads(cleanup_path.read_text())
    cleanup['deleted'].append(dict(path=str(archive),bytes=archive.stat().st_size,sha256=sha(archive)))
    cleanup_path.write_text(json.dumps(cleanup,indent=2))
    archive.unlink();marker.unlink()
previous=HOME/'results_n1_f0s0_native200_v1/summary.csv'
if previous.exists():
    old=pd.read_csv(previous,index_col=0)
    comparison=pd.DataFrame({'one_subject':old['mean_subfield_dice'],'two_subjects':summary['mean_subfield_dice']})*100
    comparison['gain_percentage_points']=comparison['two_subjects']-comparison['one_subject']
    display(comparison)
    comparison.to_csv(RESULTS/'comparison_with_one_subject.csv')
if (RESULTS/'evaluation_acceleration.json').exists():
    acceleration=json.loads((RESULTS/'evaluation_acceleration.json').read_text())
    print('Evaluation backend:',acceleration['status'],'measured slice speedup:',acceleration.get('observed_slice_speedup'),'fallback:',acceleration.get('fallback_reason'))
print('Inspect the n2 result before scheduling any other experiment.')

# Include the derived n1/n2 comparison in the final durable archive.
metadata={p.name:sha(p) for p in RESULTS.iterdir() if p.is_file() and p.suffix in ('.json','.csv','.png','.log','.jsonl') and p.name not in ('archive_manifest.json','cleanup.json')}
manifest_path.write_text(json.dumps(metadata,indent=2))
for name, expected_sha in metadata.items():
    if sha(RESULTS/name)!=expected_sha: raise ValueError('Final archive changed')
